# 01 — Data Loading and Diagnosis

This notebook covers the initial data loading, construction of the master dataframe, and the data diagnosis phase. Its output is `checkpoint_end_Data_Diagnosis.pkl`, which is the input for `02_feature_selection.ipynb`.

**Input:** `target.csv`, `X.csv`  
**Output:** `data/processed/checkpoint_end_Data_Diagnosis.pkl`

---

## Project Overview

This project focuses on the development of a complete Credit Risk Modeling framework using consumer loan data originated by Lending Club. The main objective is to estimate the total expected economic loss associated with a loan portfolio acquisition under the Basel II/III framework:

$$
\text{Expected Loss (EL)} = PD \times LGD \times EAD
$$

Where:
- **PD (Probability of Default):** likelihood that a borrower will default.
- **LGD (Loss Given Default):** percentage of exposure lost in the event of default.
- **EAD (Exposure at Default):** total exposure at the time of default (`funded_amnt`).

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

from feature_selection import diagnose_features
from feature_plot import (
    analyze_zero_variance_feature,
    analyze_categorical_feature,
    plot_continuous_variable_categoric_feature,
    plot_boxplots_list,
    plot_scatter_list
)
from feature_statistics import compare_distributions
from feature_engineering import group_emp_title, clean_emp_length

## 1. Data Load

In [ ]:
# Load target and feature CSVs and merge into a single dataframe
target_df = pd.read_csv(r"data/raw/target.csv")
X_df      = pd.read_csv(r"data/raw/X.csv")

df = pd.concat([target_df, X_df], axis=1)
df.head()

In [ ]:
# Dataset spans loans originated between 2007 and 2018
df["issue_date_year"].describe()

In [ ]:
# Default rate — approximately 13%, stratified target
print(f"Default rate: {df['y'].mean():.4f}")
print(f"Dataset shape: {df.shape}")

## 2. Master Dataframe Construction

Two independent models will be trained — a classification model for PD and a regression model for LGD. Each model operates on a different population and requires a different target variable:

- **PD target:** binary variable `Target_Variable_Default` (0/1)
- **LGD target:** `Target_Variable_Loss = 100 * max_bal_owed / funded_amnt`, computed only on defaulted loans
- **EAD:** `funded_amnt` (original loan amount at origination)

Loans with `Target_Variable_Loss >= 300%` are removed as economically implausible.

In [ ]:
# Build master dataframe with both target variables
df_master = df.copy()
df_master = df_master.rename(columns={"y": "Target_Variable_Default", "funded_amnt": "EAD"})
df_master.insert(1, "Target_Variable_Loss", 100 * (df_master["max_bal_owed"] / df_master["EAD"]))
df_master = df_master[df_master["Target_Variable_Loss"] < 300]

df_master.head()

In [ ]:
# Initialize working dataframe — df_master is kept as the unmodified control
df_master_features = df_master.copy()

# Feature lists will be populated as predictive features are identified
features_predictive_power_PD  = []
features_predictive_power_LGD = []

## 3. Data Diagnosis

Before performing deep EDA, features are screened for three conditions that may bias or artificially inflate predictive performance:

1. **High NaN percentage** (>5%): substantial information loss
2. **High cardinality with low-frequency categories**: statistical noise
3. **Low variance**: single category representing >90% of observations

Features flagged here are reviewed individually and treated via feature engineering rather than discarded automatically.

In [ ]:
# Run automated diagnosis
diagnosis = diagnose_features(df_master_features)
diagnosis

## 4. Zero Variance Features

### 4.1 `late_fees_rec`

Represents late fees received to date. 96.24% of observations are 0, but the defaulted population may behave differently.

In [ ]:
df_master_features["late_fees_rec"].value_counts()

In [ ]:
# Frequency table and distribution analysis vs PD target
analyze_zero_variance_feature(df_master_features, feature="late_fees_rec", target="Target_Variable_Default")

In [ ]:
# Statistical significance: p-value and Cohen's d
# Having late fees (binary) is highly predictive; fee amount is not
compare_distributions(df_master_features, feature="Target_Variable_Default", target="late_fees_rec", alpha=0.05)

In [ ]:
# PD: encode as binary flag — having late fees vs not
df_master_features["has_late_fees_PD"] = (df_master_features["late_fees_rec"] > 0).astype(int)
features_predictive_power_PD.append("has_late_fees_PD")

In [ ]:
# LGD: analyze predictive power over Target_Variable_Loss on defaulted population
df_master_features_default = df_master_features[df_master_features["Target_Variable_Default"] == 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df_master_features_default['late_fees_rec'],
    df_master_features_default['Target_Variable_Loss'],
    alpha=0.15, s=8, color='#3266ad'
)
ax.set_xlabel('late_fees_rec in default case', fontsize=12)
ax.set_ylabel('Target_Variable_Loss (%)', fontsize=12)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# LGD: clip at 99.9th percentile to handle long upper tail
quantile_999 = df_master_features_default['late_fees_rec'].quantile(0.999)
df_master_features["late_fees_rec_LGD"] = df_master_features["late_fees_rec"].clip(upper=quantile_999)
features_predictive_power_LGD.append("late_fees_rec_LGD")

# Drop original feature
df_master_features = df_master_features.drop(columns="late_fees_rec")

### 4.2 `disbursement_method`

Method by which the borrower receives the loan (Cash vs DirectPay). DirectPay borrowers show substantially lower default rates.

In [ ]:
analyze_categorical_feature(df_master_features, feature="disbursement_method", target="Target_Variable_Default")

In [ ]:
# PD: encode as binary flag — DirectPay vs Cash
df_master_features["is_directpay_PD"] = (df_master_features["disbursement_method"] == "DirectPay").astype(int)
features_predictive_power_PD.append("is_directpay_PD")

In [ ]:
# LGD: small effect size (Cohen's d = 0.235), no practical predictive power over loss severity
plot_continuous_variable_categoric_feature(df=df_master_features_default, feature="disbursement_method", target="Target_Variable_Loss")
compare_distributions(df=df_master_features_default, feature='disbursement_method', target='Target_Variable_Loss')

In [ ]:
# LGD: keep with renamed column for completeness — will be filtered in feature selection
df_master_features["disbursement_method_LGD"] = df_master_features["disbursement_method"]
df_master_features = df_master_features.drop(columns="disbursement_method")

## 5. High NaN Features

### 5.1 `mths_since_*` variables

NaNs in these variables are MNAR (Missing Not At Random) — they represent borrowers who have never had the corresponding delinquency event. NaNs are imputed with sentinel value 999 (meaning the event happened a long time ago or never occurred).

In [ ]:
df_master_features_months = df_master_features.copy()

# Binary flags to test whether having an event matters at all
df_master_features_months["has_bankcard_delinq"] = df_master_features_months["mths_since_recent_bankcard_delinq"].notna().astype(int)
df_master_features_months["has_revol_delinq"]    = df_master_features_months["mths_since_recent_revol_delinq"].notna().astype(int)
df_master_features_months["has_delinq"]          = df_master_features_months["mths_since_last_delinq"].notna().astype(int)
df_master_features_months["has_acc_op"]          = df_master_features_months["mths_since_last_installment_acc_op"].notna().astype(int)

In [ ]:
# Frequency tables — having/not having an event does not shift the base default rate
analyze_categorical_feature(df_master_features_months, feature="has_bankcard_delinq", target="Target_Variable_Default")
analyze_categorical_feature(df_master_features_months, feature="has_revol_delinq",    target="Target_Variable_Default")
analyze_categorical_feature(df_master_features_months, feature="has_delinq",          target="Target_Variable_Default")
analyze_categorical_feature(df_master_features_months, feature="has_acc_op",          target="Target_Variable_Default")

In [ ]:
mths_since_cols = [
    "mths_since_recent_bankcard_delinq",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq",
    "mths_since_last_installment_acc_op"
]

# Outlier check before sentinel imputation — mths_since_recent_bankcard_delinq only has 0.06% outliers
plot_boxplots_list(df=df_master_features_months, features=mths_since_cols)

In [ ]:
# PD: impute NaNs with sentinel value 999
df_master_features_months["mths_since_recent_bankcard_delinq_PD"] = df_master_features_months["mths_since_recent_bankcard_delinq"]

df_master_features["mths_since_recent_bankcard_delinq_PD"]    = df_master_features_months["mths_since_recent_bankcard_delinq_PD"].fillna(999)
df_master_features["mths_since_recent_revol_delinq_PD"]       = df_master_features_months["mths_since_recent_revol_delinq"].fillna(999)
df_master_features["mths_since_last_delinq_PD"]               = df_master_features_months["mths_since_last_delinq"].fillna(999)
df_master_features["mths_since_last_installment_acc_op_PD"]   = df_master_features_months["mths_since_last_installment_acc_op"].fillna(999)

# LGD: same imputation
df_master_features["mths_since_recent_bankcard_delinq_LGD"]   = df_master_features_months["mths_since_recent_bankcard_delinq"].fillna(999)
df_master_features["mths_since_recent_revol_delinq_LGD"]      = df_master_features_months["mths_since_recent_revol_delinq"].fillna(999)
df_master_features["mths_since_last_delinq_LGD"]              = df_master_features_months["mths_since_last_delinq"].fillna(999)
df_master_features["mths_since_last_installment_acc_op_LGD"]  = df_master_features_months["mths_since_last_installment_acc_op"].fillna(999)

# Drop original columns
df_master_features = df_master_features.drop(columns=mths_since_cols)
del df_master_features_months

### 5.2 `emp_title`

Job title supplied by borrower. High cardinality with many low-frequency categories. Grouped into 12 professional categories using a normalized mapping. NaNs imputed as 'Unknown'.

In [ ]:
df_master_features["emp_title"].value_counts().head(20)

In [ ]:
# Group emp_title into 12 high-level professional categories (min_freq=10)
df_master_features_aux = df_master_features.copy()
df_master_features_aux = group_emp_title(df_master_features_aux, col="emp_title", min_freq=10)

df_master_features["emp_title"] = df_master_features_aux["emp_title_grouped"]
del df_master_features_aux

### 5.3 `emp_length`

Employment length in years. Contains text formatting ('10+ years', '< 1 year'). NaNs treated as unemployment (imputed to 0).

In [ ]:
df_master_features["emp_length"].value_counts()

In [ ]:
# Clean emp_length: convert to numeric, cap at 10, set < 1 year to 0.5, NaN to 0
df_master_features["emp_length"] = df_master_features["emp_length"].apply(clean_emp_length)

# Use unemployment information to update emp_title
df_master_features.loc[df_master_features["emp_length"] == 0, "emp_title"] = "Unemployed"

df_master_features["emp_length"].value_counts()
df_master_features["emp_title"].value_counts(normalize=True).round(2) * 100

## 6. Checkpoint: Save Diagnosed Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_end_Data_Diagnosis.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

print(f"Saved. Shape: {df_master_features.shape}")

In [ ]:
# Load checkpoint (run this cell to skip re-running the diagnosis)
with open("data/processed/checkpoint_end_Data_Diagnosis.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]